# Reproduce LLM-MovieLens headline numbers (CIKM 2026 Full Research)

Verifies every cell in **Table 4** (main benchmark results) and every row in **Table 11** (paired t-tests) against the released per-seed metrics. 

Runs in <30 seconds. No GPU required. Prerequisites: `numpy`, `pandas`, `scipy`.

In [ ]:
# Run the verifier and print its output inline
import subprocess, sys
out = subprocess.run([sys.executable, '../reproducibility/verify_headline_numbers.py'], capture_output=True, text=True)
print(out.stdout)
if out.returncode != 0:
    print('STDERR:', out.stderr, file=sys.stderr)
    raise RuntimeError(f'Verifier exited with code {out.returncode}')

## Manually inspect a single comparison

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../reproducibility'))
from verify_headline_numbers import PER_SEED
import numpy as np
from scipy import stats

# Q2 central claim: M4 (LLM profile) > M2 (genome PCA-128d)
m4 = np.array(PER_SEED['M4']['NDCG@10'])
m2 = np.array(PER_SEED['M2']['NDCG@10'])
delta = (m4 - m2).mean()
rel = delta / m2.mean() * 100
t, p = stats.ttest_rel(m4, m2)
print(f'M4 NDCG@10: {m4.mean():.4f} ± {m4.std(ddof=1):.4f}')
print(f'M2 NDCG@10: {m2.mean():.4f} ± {m2.std(ddof=1):.4f}')
print(f'Delta: +{delta:.4f} ({rel:+.1f}%)')
print(f'Paired t-test: t={t:.2f}, p={p:.4f}')
print(f'Paper claim: M4 vs M2 = +2.5% NDCG@10, p=0.001')